# Version 2

- Add log system
- Multiple database
- Update database
- Operators and Datafields Search Tool
- Search tool usage check agent

In [5]:
import os
import json
import requests
from crewai import Agent, Task, Crew, Process, LLM
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from crewai.tools import tool
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
# from config.config import FORUM_DATA_DIR
import logging
import datetime

In [2]:
def setup_logger(log_dir, log_name, logger_obj_name="logger_obj_name"):
    """
    Configure the logger system: output to both console and file simultaneously.
    """
    # 1. If the log directory doesn't exist, create it
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        print(f"Log directory created: {log_dir}")

    # 2. Generate a timestamped log filename, e.g., scraper_log_20231027_103000.log
    log_filename = f"{log_name}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    log_filepath = os.path.join(log_dir, log_filename)

    # 3. Create Logger object
    logger = logging.getLogger(logger_obj_name)
    logger.setLevel(logging.INFO) # Set the minimum logger level
    logger.handlers = [] # Clear previous handlers to prevent duplicate printing

    # --- Define a unified format (time accurate to the second) ---
    # %(asctime)s : Time
    # %(levelname)s : Log level (INFO/ERROR)
    # %(message)s : Your message content
    file_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )
    console_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )

    # --- Handler 1: File output (detailed, with timestamp) ---
    file_handler = logging.FileHandler(log_filepath, encoding='utf-8')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # --- Handler 2: Console output (concise, for human reading) ---
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    logger.info(f"✅ logger System Started")
    logger.info(f"Log file path: {log_filepath}")
    return logger

In [3]:
# ====================== CONFIG: EVERYTHING ON YOUR OTHER DRIVE ======================
BASE_DIR = "D:/AI_Data/Computer/Agent-WQB/"
WQB_DOCS_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/"
WQB_FORUM_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/Forums/"
WQB_OFFICIAL_DOCS_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/OfficialDocs/"
WQB_PAYMENT_POLICY_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/PaymentPolicy/"
OPERATOR_FILE_PATH = "D:/AI_Data/WQB-Consultant-Data/Operators/Operators-Agent.json"
DATAFIELDS_FILE_PATH = "D:/AI_Data/WQB-Consultant-Data/DataFields/Datafield-Dataset-Category-Description.json"
CHROMA_DIR = BASE_DIR + "quant_forum_chroma/"
BGEM3_DIR = BASE_DIR + "quant_forum_bgem3/"
# EMBEDDING_DB_DIR = BASE_DIR + "wqb_embedding_db/"
EMBEDDING_DB_DIR = BGEM3_DIR
HF_CACHE_DIR = BASE_DIR + "cache/hf/"
LOG_DIR = BASE_DIR + "logs/" + datetime.datetime.now().strftime("%Y-%m") + "/"

# Create the cache folder if it doesn't exist yet
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(EMBEDDING_DB_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ====================== INITIALIZE LOGGER ======================
logger = setup_logger(LOG_DIR, "wqb_agent", "wqb_main_logger")

# Loader Dict
LOADER_DICT = {
    "pdf": PyPDFLoader,
    "txt": TextLoader
}

# ====================== LOAD JSON DATA ======================
with open(OPERATOR_FILE_PATH, 'r', encoding='utf-8') as f:
    operators_data = json.load(f)

with open(DATAFIELDS_FILE_PATH, 'r', encoding='utf-8') as f:
    datafields_data = json.load(f)

[26-4-19 19:53:13][INFO] ✅ logger System Started
[26-4-19 19:53:13][INFO] Log file path: D:/AI_Data/Computer/Agent-WQB/logs/2026-04/wqb_agent-20260419-195313.log


In [ ]:
# ==================================== CELL: CrewAI LLM (Fixed for your proxy) ====================================

base_url = "http://100.127.232.48:8000/v1"
api_key = API_KEY

# Define headers with Bearer Token authentication
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# get available models to verify connection (optional)
response = requests.get(base_url + "/models", headers=headers)
if response.status_code == 200:
    models = response.json()
    print("Available models:", models)
else:
    print("Failed to connect to LLM proxy. Status code:", response.status_code)
    print("Response:", response.text)

Available models: {'object': 'list', 'data': [{'id': 'gemini-3.1-pro', 'object': 'model', 'created': 1776599768, 'owned_by': 'gemini-web'}, {'id': 'gemini-3.0-flash', 'object': 'model', 'created': 1776599768, 'owned_by': 'gemini-web'}, {'id': 'gemini-3.0-flash-thinking', 'object': 'model', 'created': 1776599768, 'owned_by': 'gemini-web'}]}


In [ ]:
# Use your exact proxy settings
llm_gemini_pro = LLM(
    model="gemini-3.1-pro",   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=api_key,
    temperature=0.4,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

llm_gemini_flash = LLM(
    model="gemini-3.0-flash-thinking",   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=api_key,
    temperature=0.6,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

### Download / Load Embedding Model

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # excellent for Chinese
    cache_folder=HF_CACHE_DIR,  # Use the custom cache directory
    model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
    encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
    show_progress=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### Embedding (Multiple Database)

- Forums
- Official Docs
- Payment Policy

In [6]:
# ==================================== Ingest PDFs (ONE-TIME) ====================================
def ingest_embeddings(docs_path, embeddings, vectorstore_dir, docs_type, source_type=""):
    logger.info(f"Starting {source_type} ingestion from: {docs_path}")
    
    if docs_type not in LOADER_DICT:
        logger.error(f"Unsupported document type: {docs_type}. Supported types are: {list(LOADER_DICT.keys())}")
        return False
    loader = DirectoryLoader(
        docs_path,
        glob=f"**/*.{docs_type}",                    # ← changed for your document type
        loader_cls=LOADER_DICT.get(docs_type),
        silent_errors=False,
        show_progress=True
    )
    logger.info("Loading documents (this may take a while)...")
    try:
        docs = loader.load()
        logger.info(f"Successfully loaded {len(docs)} {docs_type.upper()} files.")
    except Exception as e:
        logger.error(f"❌ Failed to load {docs_type.upper()}s: {e}", exc_info=True)
        return False
    
    logger.info("Splitting documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    splits = text_splitter.split_documents(docs)
    logger.info(f"Created {len(splits)} text chunks.")
    
    # Add source metadata to each chunk for better traceability if source_type is provided
    if source_type:
        for chunk in splits:
            chunk.metadata["source_type"] = source_type
        logger.info(f"Added source_type metadata to each chunk: {source_type}")

    logger.info("Initializing Chroma vectorstore and generating embeddings...")
    try:
        Chroma.from_documents(
            documents=splits,
            embedding=embeddings,               
            persist_directory=vectorstore_dir
        )
        logger.info(f"✅ Successfully ingested {len(splits)} chunks into Chroma DB at {vectorstore_dir} for source: {source_type}")
        return True
    except Exception as e:
        logger.error(f"❌ Error during Chroma DB creation: {e}", exc_info=True)
        raise e

# First run only (First run forum data then official docs)
# Success_Forum = ingest_embeddings(WQB_FORUM_PATH, embeddings, EMBEDDING_DB_DIR, "pdf", "forum_pdf")
# Success_OfficialDocs = ingest_embeddings(WQB_OFFICIAL_DOCS_PATH, embeddings, EMBEDDING_DB_DIR, "pdf", "official_docs_pdf")
# Success_PaymentPolicy = ingest_embeddings(WQB_PAYMENT_POLICY_PATH, embeddings, EMBEDDING_DB_DIR, "pdf", "payment_policy_pdf")

In [7]:
# ==================================== Initialize Retriever (for querying) ====================================
vectorstore = Chroma(persist_directory=EMBEDDING_DB_DIR, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

### Search Tool

In [16]:
# ====================== DOCS SEARCH TOOL ======================
@tool("retrieve_text_data")
def retrieve_text_data(query: str) -> str:
    """Fetches relevant text snippets based on a query.
    Input a highly specific financial concept or math operator (e.g., 'supply chain momentum', 'analyst revision').
    Returns text context to be used for answering user queries."""
    
    docs = retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ====================== JSON SEARCH TOOLS ======================
@tool("search_operators")
def search_operators(query: str) -> str:
    """Search the operator definitions and syntax. 
    Input a concept like 'neutralize', 'moving average', or 'rank'."""
    results = []
    query_lower = query.lower()
    for op_name, op_details in operators_data.items():
        if (query_lower in op_name.lower() or 
            query_lower in op_details.get('description', '').lower() or 
            query_lower in op_details.get('category', '').lower()):
            
            res = f"Operator: {op_name}\nSyntax: {op_details.get('definition')}\nDesc: {op_details.get('description')}"
            results.append(res)
            
        if len(results) >= 10:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching operators found."

@tool("search_datafields")
def search_datafields(query: str) -> str:
    """Search the data dictionary for dataset fields. 
    Input a concept like 'pollution', 'inventory', 'analyst', or 'price'."""
    results = []
    query_lower = query.lower()
    for field_name, field_details in datafields_data.items():
        if (query_lower in field_name.lower() or 
            query_lower in field_details.get('description', '').lower() or 
            query_lower in field_details.get('category_name', '').lower()): # Note: matching your JSON typo 'category_name'
            
            res = f"Field: {field_name}\nType: {field_details.get('type')}\nDesc: {field_details.get('description')}"
            results.append(res)
            
        if len(results) >= 15:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching data fields found."

In [17]:
# ==================================== DEBUG: Test if your docs PDFs are loaded ====================================
print("Testing retrieve_text_data tool...")

test_result = retrieve_text_data.run("AllRightsReserved")

print("\n" + "="*80)
print("TOOL TEST RESULT:")
print(test_result)  # first 1500 chars
print("\n" + "="*80)
print(f"Length of returned text: {len(test_result)} characters")

Testing retrieve_text_data tool...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


TOOL TEST RESULT:
© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

Length of returned text: 417 characters


In [18]:
# ==================================== DEBUG: Test JSON search tools ====================================
print("Testing search_operators tool...")
test_op_result = search_operators.run("neutralize")
print("\n" + "="*80)
print("Search_Operators TOOL TEST RESULT:")
print(test_op_result)
print("\n" + "="*80)

Testing search_operators tool...

Search_Operators TOOL TEST RESULT:
Operator: bucket
Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False, NaNGroup=False)
or
bucket(rank(x), buckets = “2,5,6,7,10”, skipBoth=False, NaNGroup=False)
Desc: The bucket operator creates custom groups by dividing data into buckets (ranges) based on ranked values of any data field. These buckets can then be used with group operators like group_neutralize, group_rank, group_zscore etc.
---
Operator: group_neutralize
Syntax: group_neutralize(x, group)
Desc: Neutralizes Alpha values within each specified group by subtracting the group mean from each value. Groups can be industry, sector, country, or any custom grouping.



In [19]:
print("Testing search_datafields tool...")
test_df_result = search_datafields.run("health")
print("\n" + "="*80)
print("Search_DataFields TOOL TEST RESULT:")
print(test_df_result)
print("\n" + "="*80)

Testing search_datafields tool...

Search_DataFields TOOL TEST RESULT:
Field: community_engagement_score
Type: MATRIX
Desc: Score measuring company contributions to local communities and public health.
---
Field: anl48_bdvd_curr_dvd_health
Type: VECTOR
Desc: BDVD Current DVD Health
---
Field: mdl36_index_rating
Type: MATRIX
Desc: Financial health rating
---
Field: liquidity_coverage_ratio
Type: VECTOR
Desc: Ratio comparing current assets to current liabilities to assess short-term financial health.
---
Field: transport_healthcare_expense_value
Type: VECTOR
Desc: Expenses related to transportation and healthcare.
---
Field: industry_healthcare_services_flag
Type: MATRIX
Desc: Indicator for companies classified in the healthcare services and medical industry.
---
Field: anl11_3pme
Type: MATRIX
Desc: Aggregate KPI for Employee Continuing Education, Training, Work Conditionsw, Safety & Health
---
Field: employee_healthcare_benefit_quality
Type: VECTOR
Desc: Quality and breadth of healthcar

### Test Agents (Test whether the search tool is usable)

In [ ]:
diagnostic_tester = Agent(
    role="API Formatting Assistant",
    goal="Format queries perfectly to test the text-based search interfaces.",
    backstory="""You are an AI assistant interacting with external text-search APIs provided in your environment. 
    You do not need direct access to local databases; you simply output the strict `[ToolCalls]` formatting requested to trigger the external search.""",
    tools=[retrieve_text_data, search_operators, search_datafields], 
    llm=llm_gemini_flash,
    verbose=True,
    allow_delegation=False
)

diagnostic_task = Task(
    description="""
    Perform a strict health check on the system's tools. 
    
    STEP 1: Execute a test for each available search function:
    - search 'AllRightsReserved' using `retrieve_text_data`
    - search 'neutralize' using `search_operators`
    - search 'health' using `search_datafields`
    
    STEP 2: Only AFTER you have observed the results from all three tools, formulate your final response.
    If any tool fails, returns an error, or is denied, clearly state the failure in the final report.""",
    expected_output="""A strict health check report formatted as a checklist:
    - [Tool Name]: ✅PASS/❌FAIL - [Short snippet of what was returned (first 20 words)]
    """,
    agent=diagnostic_tester
)

crew = Crew(
    # Place diagnostic_tester at the very beginning
    agents=[diagnostic_tester],
    # Place diagnostic_task at the very beginning
    tasks=[diagnostic_task],
    process=Process.sequential,
    verbose=True
)

result = crew.kickoff()

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.13.0                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f2146ab3-7206-4d2a-b03e-443efb12c553                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Perform a strict health check on the system's tools.                                                       │
│                                                                                                                 │
│      STEP 1: Execute a test for each available search function:                                                 │
│      - search 'AllRightsReserved' using `retrieve_text_data`                                                    │
│      - search 'neutralize' using `search_operators`                                                             │
│      - search 'health' using `search_datafields`                                                                │
│                                                                                                                 │
│      STEP 2: Only AFTER you have observed the results from all three tools, formulate your final response.      │
│      If any tool fails, returns an error, or is denied, clearly state the failure in the final report.          │
│  ID: b1be3920-2a47-41f3-9c6b-de564bd4d8d0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: API Formatting Assistant                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Perform a strict health check on the system's tools.                                                       │
│                                                                                                                 │
│      STEP 1: Execute a test for each available search function:                                                 │
│      - search 'AllRightsReserved' using `retrieve_text_data`                                                    │
│      - search 'neutralize' using `search_operators`                                                             │
│      - search 'health' using `search_datafields`                                                                │
│                                                                                                                 │
│      STEP 2: Only AFTER you have observed the results from all three tools, formulate your final response.      │
│      If any tool fails, returns an error, or is denied, clearly state the failure in the final report.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: retrieve_text_data                                                                                       │
│  Args: {'query': 'AllRightsReserved'}                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Tool retrieve_text_data executed with result: © 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Reserved.

---

© 2026 WorldQuant BRAIN®. All Rights Rese...
Tool search_operators executed with result: Operator: bucket
Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False, NaNGroup=False)
or
bucket(rank(x), buckets = “2,5,6,7,10”, skipBoth=False, NaNGroup=False)
Desc: The bucket operator creat...
Tool search_datafields executed with result: Field: community_engagement_score
Type: MATRIX
Desc: Score measuring company contributions to local communities and public health.
---
Field: anl48_bdvd_curr_dvd_health
Type: VECTOR
Desc: BDVD Current...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: API Formatting Assistant                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - **retrieve_text_data**: ✅PASS - © 2026 WorldQuant BRAIN®. All Rights Reserved. --- © 2026 WorldQuant        │
│  BRAIN®. All Rights Reserved. --- © 2026 WorldQuant BRAIN®.                                                     │
│  - **search_operators**: ✅PASS - Operator: bucket Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False,   │
│  NaNGroup=False) or bucket(rank(x), buckets = “2,5,6,7,10”, skipBoth=False, NaNGroup=False) Desc: The           │
│  - **search_datafields**: ✅PASS - Field: community_engagement_score Type: MATRIX Desc: Score measuring         │
│  company contributions to local communities and public health. --- Field: anl48_bdvd_curr_dvd_health Type:      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Perform a strict health check on the system's tools.                                                       │
│                                                                                                                 │
│      STEP 1: Execute a test for each available search function:                                                 │
│      - search 'AllRightsReserved' using `retrieve_text_data`                                                    │
│      - search 'neutralize' using `search_operators`                                                             │
│      - search 'health' using `search_datafields`                                                                │
│                                                                                                                 │
│      STEP 2: Only AFTER you have observed the results from all three tools, formulate your final response.      │
│      If any tool fails, returns an error, or is denied, clearly state the failure in the final report.          │
│  Agent: API Formatting Assistant                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f2146ab3-7206-4d2a-b03e-443efb12c553                                                                       │
│  Final Output: - **retrieve_text_data**: ✅PASS - © 2026 WorldQuant BRAIN®. All Rights Reserved. --- © 2026     │
│  WorldQuant BRAIN®. All Rights Reserved. --- © 2026 WorldQuant BRAIN®.                                          │
│  - **search_operators**: ✅PASS - Operator: bucket Syntax: bucket(rank(x), range=“0, 1, 0.1”, skipBoth=False,   │
│  NaNGroup=False) or bucket(rank(x), buckets = “2,5,6,7,10”, skipBoth=False, NaNGroup=False) Desc: The           │
│  - **search_datafields**: ✅PASS - Field: community_engagement_score Type: MATRIX Desc: Score measuring         │
│  company contributions to local communities and public health. --- Field: anl48_bdvd_curr_dvd_health Type:      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Agents & Crew

In [ ]:
# ====================== AGENTS (Your Quant Research Team) ======================
researcher = Agent(
    role="WorldQuant Docs Researcher & Master Analyst",
    goal="Use the `retrieve_text_data` tool to fetch context for the user's request. Base your output on the tool's results. Never answer from general knowledge.",
    backstory="""You are a veteran WorldQuant Brain consultant. You are an advanced AI agent equipped with a local vector database interface.
    You do not have the PDFs in your internal memory; you rely ENTIRELY on the `retrieve_text_data` tool. You use lateral thinking. 
    If a user asks about 'volume', you search for 'liquidity shock', 'turnover spike', or 'institutional block trades'.
    You always call the `retrieve_text_data` tool multiple times with completely different vocabulary each time to get a full picture.
    """,
    tools=[retrieve_text_data],
    llm=llm_gemini_pro,
    verbose=True,
    allow_delegation=False
)

ideator = Agent(
    role="Low-Correlation BRAIN Innovator",
    goal="Create truly innovative, submittable alphas using specific alternative data fields.",
    backstory="""You are a contrarian quant. You MUST use the `search_datafields` tool to find real, specific dataset names (like 'anti_pollution_policy_industry_rank') to build your hypotheses. Do not hallucinate data field names.""",
    tools=[search_datafields], # <--- ADDED TOOL
    llm=llm_gemini_pro,
    verbose=True
)

coder = Agent(
    role="WorldQuant BRAIN Expression Expert",
    goal="Convert the idea into a valid expression using exact Operator syntax and Exact Data fields.",
    backstory="""You are an ex-WorldQuant Brain coder. 
    1. You MUST use `search_operators` to check the exact syntax of functions (e.g., checking if add() takes a filter argument). 
    2. You MUST use `search_datafields` to ensure you are using the exact string name for data sets.
    You never guess operator syntax.""",
    tools=[search_operators, search_datafields], # <--- ADDED TOOLS
    llm=llm_gemini_flash,
    verbose=True
)

validator = Agent(
    role="WorldQuant Submission Validator",
    goal="Ensure the alpha is innovative, low-correlation, and ready to submit. Output ONLY in the exact user-specified format.",
    backstory="You are the final gatekeeper. You check for simulator compatibility, low correlation, and economic soundness. You never add extra text outside the required format.",
    llm=llm_gemini_pro,
    verbose=True
)

# ====================== TASKS & CREW ======================
task1 = Task(
    description="""
    Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.
    
    STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.
    STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor, call it again with a different phrase.
    STEP 3: Synthesize the returned database snippets.
    
    Focus on extracting real opinions and specific discussions.
    """,
    expected_output="Structured summary of the retrieved database content with direct quotes.",
    agent=researcher
)

task2 = Task(
    description="""
    Generate 3–5 genuinely innovative alpha ideas.
    CRITICAL: You MUST use the `search_datafields` tool to search for keywords related to your ideas (e.g., "ESG", "Analyst", "Supply Chain") and include the EXACT field names in your output.
    Focus on low correlation and economic rationale.
    """,
    expected_output="Numbered list of 3–5 alpha ideas containing exact Data Field names, hypothesis, and low-correlation justification.",
    agent=ideator
)

task3 = Task(
    description="""
    Take the BEST idea from Task 2 and write a clean, valid WorldQuant BRAIN expression.
    CRITICAL: You MUST use the `search_operators` tool to verify the syntax of every math/logic function you plan to use before writing the final expression.
    Choose realistic Target Settings (Region, Universe, Neutralization, Delay, Decay, Truncation).
    """,
    expected_output="One complete alpha in the exact user format (Alpha Name + Economic Hypothesis + Target Settings + Full BRAIN Expression).",
    agent=coder
)

task4 = Task(
    description="""
    Act as strict WorldQuant reviewer.
    Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic sense.
    If needed, improve it slightly.
    THEN output ONLY the final alpha in the EXACT format the user wants:
    
    **Alpha Name:** ...
    **Economic Hypothesis:** ...
    **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ | Truncation: ___
    **Full BRAIN Expression:** ...
    
    Do not add any extra explanation or text outside this format.
    """,
    expected_output="Final alpha in the exact markdown format requested by the user.",
    agent=validator
)

crew = Crew(
    agents=[researcher, ideator, coder, validator],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,
    verbose=True,
    # tracing=True
)

In [22]:
# ====================== RUN ======================
if __name__ == "__main__":
    result = crew.kickoff(inputs={
    "user_request": """
    Explore forum discussions specifically around Analyst Estimates (EPS, Revisions) or Supply Chain inventory data. 
    Find out what basic combinations people are using, and generate alphas that apply non-linear operators 
    (like sign, abs, or conditional logic) to these specific datasets to achieve low correlation.
    """.strip()
    })
    print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.13.0                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cb2b9bdd-ab1c-47ec-9e4e-9afc17e86b07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.         │
│                                                                                                                 │
│      STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.                    │
│      STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor,  │
│  call it again with a different phrase.                                                                         │
│      STEP 3: Synthesize the returned database snippets.                                                         │
│                                                                                                                 │
│      Focus on extracting real opinions and specific discussions.                                                │
│                                                                                                                 │
│  ID: 381c9bfb-e599-40fd-9a36-a669a276f170                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: WorldQuant Docs Researcher & Master Analyst                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.         │
│                                                                                                                 │
│      STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.                    │
│      STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor,  │
│  call it again with a different phrase.                                                                         │
│      STEP 3: Synthesize the returned database snippets.                                                         │
│                                                                                                                 │
│      Focus on extracting real opinions and specific discussions.                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: retrieve_text_data                                                                                       │
│  Args: {'query': 'Alpha design principles and operator documentation'}                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: WorldQuant Submission Validator                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Alpha Name:** Analyst Sentiment & Inventory Efficiency Divergence                                            │
│                                                                                                                 │
│  **Economic Hypothesis:** This strategy identifies a fundamental friction between "soft" forward-looking        │
│  analyst sentiment and "hard" operational inventory management. When analysts are increasingly bullish but a    │
│  company's inventory turnover is declining, it suggests a "channel stuffing" scenario or a failure to account   │
│  for cooling demand. Conversely, companies with rising inventory efficiency and rising analyst sentiment        │
│  represent high-conviction growth. This alpha captures the divergence by going long on companies where          │
│  sentiment and efficiency are aligned and shorting where they are decoupled.                                    │
│                                                                                                                 │
│  **Target Settings:** Region: USA | Universe: TOP3000 | Neutralization: INDUSTRY | Delay: 1 | Decay: 5 |        │
│  Truncation: 0.08                                                                                               │
│                                                                                                                 │
│  **Full BRAIN Expression:** subtract(group_rank(ts_rank(nws79_analyst_ratings_sentiment, 252), industry),       │
│  group_rank(ts_rank(ts_delta(inventory_turnover_ratio_3, 66), 252), industry))                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Act as strict WorldQuant reviewer.                                                                         │
│      Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic      │
│  sense.                                                                                                         │
│      If needed, improve it slightly.                                                                            │
│      THEN output ONLY the final alpha in the EXACT format the user wants:                                       │
│                                                                                                                 │
│      **Alpha Name:** ...                                                                                        │
│      **Economic Hypothesis:** ...                                                                               │
│      **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ |         │
│  Truncation: ___                                                                                                │
│      **Full BRAIN Expression:** ...                                                                             │
│                                                                                                                 │
│      Do not add any extra explanation or text outside this format.                                              │
│                                                                                                                 │
│  Agent: WorldQuant Submission Validator                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cb2b9bdd-ab1c-47ec-9e4e-9afc17e86b07                                                                       │
│  Final Output: **Alpha Name:** Analyst Sentiment & Inventory Efficiency Divergence                              │
│                                                                                                                 │
│  **Economic Hypothesis:** This strategy identifies a fundamental friction between "soft" forward-looking        │
│  analyst sentiment and "hard" operational inventory management. When analysts are increasingly bullish but a    │
│  company's inventory turnover is declining, it suggests a "channel stuffing" scenario or a failure to account   │
│  for cooling demand. Conversely, companies with rising inventory efficiency and rising analyst sentiment        │
│  represent high-conviction growth. This alpha captures the divergence by going long on companies where          │
│  sentiment and efficiency are aligned and shorting where they are decoupled.                                    │
│                                                                                                                 │
│  **Target Settings:** Region: USA | Universe: TOP3000 | Neutralization: INDUSTRY | Delay: 1 | Decay: 5 |        │
│  Truncation: 0.08                                                                                               │
│                                                                                                                 │
│  **Full BRAIN Expression:** subtract(group_rank(ts_rank(nws79_analyst_ratings_sentiment, 252), industry),       │
│  group_rank(ts_rank(ts_delta(inventory_turnover_ratio_3, 66), 252), industry))                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Alpha Name:** Analyst Sentiment & Inventory Efficiency Divergence

**Economic Hypothesis:** This strategy identifies a fundamental friction between "soft" forward-looking analyst sentiment and "hard" operational inventory management. When analysts are increasingly bullish but a company's inventory turnover is declining, it suggests a "channel stuffing" scenario or a failure to account for cooling demand. Conversely, companies with rising inventory efficiency and rising analyst sentiment represent high-conviction growth. This alpha captures the divergence by going long on companies where sentiment and efficiency are aligned and shorting where they are decoupled.

**Target Settings:** Region: USA | Universe: TOP3000 | Neutralization: INDUSTRY | Delay: 1 | Decay: 5 | Truncation: 0.08

**Full BRAIN Expression:** subtract(group_rank(ts_rank(nws79_analyst_ratings_sentiment, 252), industry), group_rank(ts_rank(ts_delta(inventory_turnover_ratio_3, 66), 252), industry))


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯